# Heterogeneous Federated Learning — FedAvg vs Knowledge Distillation
### Thesis Implementation | Flower (flwr) | arXiv:2307.10616

## Dataset: CIFAR-10 (Real Object Images)
60,000 real colour photos across 10 object categories:

| Label | Class | Label | Class |
|---|---|---|---|
| 0 | ✈️ Airplane | 5 | 🐕 Dog |
| 1 | 🚗 Automobile | 6 | 🐸 Frog |
| 2 | 🐦 Bird | 7 | 🐴 Horse |
| 3 | 🐱 Cat | 8 | 🚢 Ship |
| 4 | 🦌 Deer | 9 | 🚛 Truck |

## Experiments
| # | Models | Method | Expected |
|---|---|---|---|
| E1 | Same (Medium CNN) | FedAvg — Homogeneous | ✅ Works |
| E2 | Different (S/M/L) | FedAvg — Heterogeneous | 💥 Crashes |
| E3 | Different (S/M/L) | Distillation — Heterogeneous | ✅ Works |

**Run Cell 1 first, then Run All.**

In [ ]:
import sys
!{sys.executable} -m pip install -q flwr torch torchvision matplotlib numpy
print('✅ All packages installed!')

---
## Section 1: Imports & Device Setup

In [ ]:
import flwr as fl
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision, torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device  : {DEVICE}')
print(f'✅ Flower  : {fl.__version__}')
print(f'✅ PyTorch : {torch.__version__}')

---
## Section 2: Configuration

In [ ]:
CONFIG = {
    'num_clients'    : 3,
    'num_rounds'     : 10,
    'local_epochs'   : 5,
    'batch_size'     : 64,
    'learning_rate'  : 0.001,
    'non_iid_alpha'  : 0.5,   # Dirichlet alpha — lower = more Non-IID
    'temperature'    : 4.0,   # distillation temperature
    'distill_weight' : 0.6,
    'distill_epochs' : 10,
    'num_public'     : 500,   # proxy samples for distillation
}
CIFAR10_CLASSES = ['Airplane','Automobile','Bird','Cat','Deer',
                   'Dog','Frog','Horse','Ship','Truck']
print('✅ Configuration loaded')
for k, v in CONFIG.items(): print(f'   {k}: {v}')

---
## Section 3: CIFAR-10 Dataset — Real Object Images

**CIFAR-10** contains 60,000 real colour photographs (32×32 pixels) across 10 classes.
This is a standard computer vision benchmark used in hundreds of FL papers.

We simulate **Non-IID statistical heterogeneity** using Dirichlet distribution (α=0.5):
each client (hospital/phone/IoT) sees a skewed subset of object categories.

In [ ]:
np.random.seed(42); torch.manual_seed(42)

# CIFAR-10 transform — standard normalization for colour images
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),       # data augmentation
    transforms.RandomCrop(32, padding=4),    # data augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010))
])

# Download CIFAR-10 automatically
train_data = torchvision.datasets.CIFAR10('.', train=True,  download=True, transform=transform_train)
test_data  = torchvision.datasets.CIFAR10('.', train=False, download=True, transform=transform_test)

def dirichlet_split(dataset, num_clients, alpha=0.5):
    """Non-IID Dirichlet split — Hsu et al. (2019). Standard FL benchmark."""
    labels = np.array(dataset.targets)
    client_indices = [[] for _ in range(num_clients)]
    for cls in range(10):
        idx = np.where(labels == cls)[0]; np.random.shuffle(idx)
        props = np.random.dirichlet(np.repeat(alpha, num_clients))
        splits = (props * len(idx)).astype(int)
        splits[-1] = len(idx) - splits[:-1].sum()
        start = 0
        for i, count in enumerate(splits):
            client_indices[i].extend(idx[start:start+count].tolist()); start += count
    return client_indices

client_indices = dirichlet_split(train_data, CONFIG['num_clients'], CONFIG['non_iid_alpha'])

public_loader  = DataLoader(Subset(test_data, list(range(CONFIG['num_public']))),
                            batch_size=CONFIG['num_public'], shuffle=False)
test_loader    = DataLoader(test_data, batch_size=256, shuffle=False)
client_loaders = [DataLoader(Subset(train_data, client_indices[i]),
                             batch_size=CONFIG['batch_size'], shuffle=True)
                  for i in range(CONFIG['num_clients'])]

print(f'✅ CIFAR-10: {len(train_data)} train | {len(test_data)} test | 10 object classes')
print(f'✅ Classes: {CIFAR10_CLASSES}')
print(f'✅ Non-IID Dirichlet split (alpha={CONFIG["non_iid_alpha"]}):')
for i, idx in enumerate(client_indices):
    lbls = np.array(train_data.targets)[idx]
    u, c = np.unique(lbls, return_counts=True)
    class_dist = {CIFAR10_CLASSES[k]: int(v) for k,v in zip(u,c)}
    print(f'   Client {i+1}: {len(idx)} samples | {class_dist}')
print(f'✅ Public proxy for distillation: {CONFIG["num_public"]} samples')

---
## Section 3b: Visualize the Dataset
Show sample images from CIFAR-10 — to present to your professor.

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('CIFAR-10 Dataset — One Sample per Class\n(Real object images used for Federated Learning)',
             fontsize=13, fontweight='bold')

# Unnormalize for display
mean = np.array([0.4914, 0.4822, 0.4465])
std  = np.array([0.2023, 0.1994, 0.2010])

# Get one sample per class from raw dataset (no transform)
raw_data = torchvision.datasets.CIFAR10('.', train=True, download=False)
shown = {}
for img, label in raw_data:
    if label not in shown:
        shown[label] = img
    if len(shown) == 10:
        break

emojis = ['✈️','🚗','🐦','🐱','🦌','🐕','🐸','🐴','🚢','🚛']
for cls in range(10):
    ax = axes[cls//5][cls%5]
    ax.imshow(shown[cls])
    ax.set_title(f'{emojis[cls]} {CIFAR10_CLASSES[cls]}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('cifar10_dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dataset sample saved as cifar10_dataset_samples.png')

# Show Non-IID distribution
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4))
fig2.suptitle('Non-IID Data Distribution Across Clients (Dirichlet α=0.5)\n'
              'Each client sees different amounts of each object class',
              fontsize=12, fontweight='bold')
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i, idx in enumerate(client_indices):
    lbls = np.array(train_data.targets)[idx]
    u, c = np.unique(lbls, return_counts=True)
    full_counts = [0]*10
    for cls, cnt in zip(u, c): full_counts[cls] = cnt
    axes2[i].bar(CIFAR10_CLASSES, full_counts, color=colors)
    axes2[i].set_title(f'Client {i+1}\n({len(idx)} total samples)', fontweight='bold')
    axes2[i].set_xlabel('Object Class'); axes2[i].set_ylabel('Samples')
    axes2[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('cifar10_noniid_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution chart saved as cifar10_noniid_distribution.png')

---
## Section 4: Asymmetric CNN Architectures

CIFAR-10 uses **colour images (3 channels, 32×32)** — so models use `Conv2d(3, ...)` instead of `Conv2d(1, ...)`.

| Client | Real-world device | Architecture | Parameters |
|---|---|---|---|
| 1 | IoT Sensor | Small CNN (1 conv) | ~15K |
| 2 | Mobile Phone | Medium CNN (2 conv) | ~100K |
| 3 | Hospital Server | Large CNN (3 conv + BN) | ~400K |

In [ ]:
class SmallCNN(nn.Module):
    """Client 1 — IoT Sensor. 1 conv layer, minimal memory."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(16*14*14, 64), nn.ReLU(),
            nn.Linear(64, 10)
        )
    def forward(self, x): return self.net(x)

class MediumCNN(nn.Module):
    """Client 2 — Mobile Phone. 2 conv layers, moderate capacity."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

class LargeCNN(nn.Module):
    """Client 3 — Hospital Server. 3 conv layers + BatchNorm, high capacity."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)

MODEL_CLASSES = [SmallCNN, MediumCNN, LargeCNN]
CLIENT_NAMES  = ['Client 1 (Small CNN — IoT Sensor)',
                 'Client 2 (Medium CNN — Mobile Phone)',
                 'Client 3 (Large CNN — Hospital Server)']

print('✅ Asymmetric CNN Architectures for CIFAR-10:')
for cls, name in zip(MODEL_CLASSES, CLIENT_NAMES):
    n = sum(p.numel() for p in cls().parameters())
    print(f'   {name}: {n:,} parameters')
print()
print('⚠️  FedAvg CANNOT average these — weight shapes are different!')
print('   Small  conv1: Conv2d(3,16,5)  → weight shape [16,3,5,5]')
print('   Medium conv1: Conv2d(3,32,3)  → weight shape [32,3,3,3]')
print('   Large  conv1: Conv2d(3,32,3)  → weight shape [32,3,3,3] + BatchNorm')
print('   → Averaging [16,3,5,5] with [32,3,3,3] = 💥 size mismatch!')

---
## Section 5: Helper Functions

In [ ]:
def train_local(model, loader, optimizer, epochs, device,
                public_X=None, soft_labels=None, config=None):
    """Local training: hard labels on private data + soft label distillation from server."""
    model.train()
    ce = nn.CrossEntropyLoss()
    kl = nn.KLDivLoss(reduction='batchmean')
    T  = config['temperature'] if config else 4.0
    dw = config['distill_weight'] if config else 0.6
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            ce(model(imgs), labels).backward()
            optimizer.step()
        if public_X is not None and soft_labels is not None:
            optimizer.zero_grad()
            logits = model(public_X.to(device))
            s_log  = F.log_softmax(logits / T, dim=1)
            t_soft = F.softmax(soft_labels.to(device) / T, dim=1)
            hard   = ce(logits, t_soft.argmax(dim=1))
            soft   = kl(s_log, t_soft) * (T**2)
            ((1-dw)*hard + dw*soft).backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def get_soft_preds(model, public_X, device):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(public_X.to(device)), dim=1).cpu()

print('✅ Helper functions ready')

---
## Experiment 1: FedAvg — Homogeneous (same model, all clients)

**All 3 clients use MediumCNN.** The server averages weights — this works because all shapes match.
This is the standard FedAvg setting (McMahan et al., 2017).

In [ ]:
print('='*60)
print('  E1: FedAvg — Homogeneous (same MediumCNN on all clients)')
print('='*60)

# All clients use SAME architecture
fa_clients  = [MediumCNN().to(DEVICE) for _ in range(CONFIG['num_clients'])]
fa_opts     = [optim.Adam(m.parameters(), lr=CONFIG['learning_rate']) for m in fa_clients]
fa_global   = MediumCNN().to(DEVICE)
fedavg_history = {'round':[], 'global_acc':[], 'client_accs':[]}

for rnd in range(1, CONFIG['num_rounds']+1):
    # Distribute global model weights to all clients (only works with same architecture)
    g_state = fa_global.state_dict()
    for c in fa_clients: c.load_state_dict(g_state)

    # Local training on private CIFAR-10 data
    for c, opt, loader in zip(fa_clients, fa_opts, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

    # FedAvg: weighted average of weights by dataset size
    total = sum(len(client_indices[i]) for i in range(CONFIG['num_clients']))
    avg_state = {}
    for key in fa_clients[0].state_dict():
        avg_state[key] = sum(
            fa_clients[i].state_dict()[key].float() * (len(client_indices[i]) / total)
            for i in range(CONFIG['num_clients'])
        )
    fa_global.load_state_dict(avg_state)

    g_acc  = evaluate_model(fa_global, test_loader, DEVICE)
    c_accs = [evaluate_model(c, test_loader, DEVICE) for c in fa_clients]
    fedavg_history['round'].append(rnd)
    fedavg_history['global_acc'].append(g_acc)
    fedavg_history['client_accs'].append(c_accs)
    print(f'  Round {rnd:02d} | Global: {g_acc:.4f} | Clients: {[f"{a:.4f}" for a in c_accs]}')

print(f'\n✅ E1 Final Global Accuracy: {fedavg_history["global_acc"][-1]:.4f}')

---
## Experiment 2: FedAvg — Heterogeneous (different models) → PROOF IT FAILS

Now clients have **different architectures** (Small/Medium/Large CNN).
FedAvg will crash every round because weight shapes are incompatible.
This is the mathematical proof that FedAvg cannot handle model heterogeneity.

In [ ]:
print('='*60)
print('  E2: FedAvg — Heterogeneous (different CNNs) — EXPECT CRASH')
print('='*60)

# Each client has a DIFFERENT architecture — this is the real-world scenario
fah_clients = [cls().to(DEVICE) for cls in MODEL_CLASSES]
fah_opts    = [optim.Adam(m.parameters(), lr=CONFIG['learning_rate']) for m in fah_clients]
fah_global  = MediumCNN().to(DEVICE)
fedavg_hetero_history = {'round':[], 'global_acc':[], 'client_accs':[], 'crashed': []}

for rnd in range(1, CONFIG['num_rounds']+1):
    # Local training — each client trains its own architecture
    for c, opt, loader in zip(fah_clients, fah_opts, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

    # FedAvg aggregation — will CRASH because shapes differ
    crashed = False
    try:
        total = sum(len(client_indices[i]) for i in range(CONFIG['num_clients']))
        avg_state = {}
        for key in fah_clients[0].state_dict():
            avg_state[key] = sum(
                fah_clients[i].state_dict()[key].float() * (len(client_indices[i]) / total)
                for i in range(CONFIG['num_clients'])
            )
        fah_global.load_state_dict(avg_state)
        g_acc = evaluate_model(fah_global, test_loader, DEVICE)
    except Exception as e:
        print(f'  💥 Round {rnd:02d}: FedAvg CRASHED!')
        print(f'     Error: {str(e)[:80]}')
        g_acc   = 0.0
        crashed = True

    c_accs = [evaluate_model(c, test_loader, DEVICE) for c in fah_clients]
    fedavg_hetero_history['round'].append(rnd)
    fedavg_hetero_history['global_acc'].append(g_acc)
    fedavg_hetero_history['client_accs'].append(c_accs)
    fedavg_hetero_history['crashed'].append(crashed)
    if not crashed:
        print(f'  Round {rnd:02d} | Global: {g_acc:.4f}')

crashes = sum(fedavg_hetero_history['crashed'])
print(f'\n❌ E2 Result: FedAvg crashed in {crashes}/{CONFIG["num_rounds"]} rounds')
print(f'   Global accuracy stuck at: {fedavg_hetero_history["global_acc"][-1]:.4f} (random = 0.1)')

---
## Experiment 3: Ensemble Distillation — Heterogeneous (different models) → WORKS

Same 3 clients with different architectures, but now using **Ensemble Knowledge Distillation**.
Clients share only **soft predictions** on a public proxy set — no weight averaging needed.
This is **FedDF** (Lin et al., 2020) — the federated transfer learning approach.

In [ ]:
print('='*60)
print('  E3: Distillation — Heterogeneous (different CNNs) — SHOULD WORK')
print('='*60)

# Same asymmetric clients as E2
dist_clients = [cls().to(DEVICE) for cls in MODEL_CLASSES]
dist_opts    = [optim.Adam(m.parameters(), lr=CONFIG['learning_rate']) for m in dist_clients]
server_model = MediumCNN().to(DEVICE)
server_opt   = optim.Adam(server_model.parameters(), lr=CONFIG['learning_rate'])
public_X, _  = next(iter(public_loader))
server_soft  = None
distill_history = {'round':[], 'global_acc':[], 'client_accs':[]}
kl = nn.KLDivLoss(reduction='batchmean')
ce = nn.CrossEntropyLoss()
T  = CONFIG['temperature']

for rnd in range(1, CONFIG['num_rounds']+1):
    # Step 1: Each client trains locally (private data + server knowledge)
    for c, opt, loader in zip(dist_clients, dist_opts, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE,
                    public_X=public_X, soft_labels=server_soft, config=CONFIG)

    # Step 2: Clients send ONLY soft predictions (not raw data, not weights)
    soft_preds = [get_soft_preds(c, public_X, DEVICE) for c in dist_clients]

    # Step 3: Server ensembles predictions
    ensemble = torch.stack(soft_preds).mean(dim=0)
    pseudo   = ensemble.argmax(dim=1).to(DEVICE)

    # Step 4: Server distills ensemble into global model
    server_model.train()
    for _ in range(CONFIG['distill_epochs']):
        server_opt.zero_grad()
        logits = server_model(public_X.to(DEVICE))
        s_log  = F.log_softmax(logits / T, dim=1)
        t_soft = F.softmax(ensemble.to(DEVICE) / T, dim=1)
        (0.4*ce(logits, pseudo) + 0.6*kl(s_log, t_soft)*(T**2)).backward()
        server_opt.step()

    # Step 5: Broadcast global knowledge back to clients as soft labels
    server_model.eval()
    with torch.no_grad():
        server_soft = F.softmax(server_model(public_X.to(DEVICE)), dim=1).cpu()

    g_acc  = evaluate_model(server_model, test_loader, DEVICE)
    c_accs = [evaluate_model(c, test_loader, DEVICE) for c in dist_clients]
    distill_history['round'].append(rnd)
    distill_history['global_acc'].append(g_acc)
    distill_history['client_accs'].append(c_accs)
    print(f'  Round {rnd:02d} | Global: {g_acc:.4f} | Clients: {[f"{a:.4f}" for a in c_accs]}')

print(f'\n✅ E3 Final Global Accuracy: {distill_history["global_acc"][-1]:.4f}')

---
## Section 9: Results Visualization — All 3 Experiments

In [ ]:
rounds = list(range(1, CONFIG['num_rounds']+1))
colors_clients = ['#e07b54','#5b8db8','#6bbf6a']
cnames = ['Client 1 (Small CNN)', 'Client 2 (Medium CNN)', 'Client 3 (Large CNN)']

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(
    'Heterogeneous Federated Learning on CIFAR-10 (Real Object Images)\n'
    'FedAvg vs Ensemble Distillation | Flower Framework | arXiv:2307.10616',
    fontsize=14, fontweight='bold'
)

# Plot 1: Global model comparison
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(rounds, fedavg_history['global_acc'], 'b-o', lw=2, ms=6, label='E1: FedAvg Homo')
ax1.plot(rounds, fedavg_hetero_history['global_acc'], 'r-x', lw=2, ms=8, label='E2: FedAvg Hetero (💥)')
ax1.plot(rounds, distill_history['global_acc'], 'g-s', lw=2, ms=6, label='E3: Distillation Hetero')
ax1.set(xlabel='Round', ylabel='Test Accuracy', title='Global Model Accuracy\nAll 3 Experiments')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3); ax1.set_ylim(0, 1.05)

# Plot 2: E1 FedAvg clients
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(rounds, fedavg_history['global_acc'], 'k-o', lw=2.5, ms=7, label='Global', zorder=5)
for i, (name, color) in enumerate(zip(cnames, colors_clients)):
    ax2.plot(rounds, [r[i] for r in fedavg_history['client_accs']], '--', lw=1.5, color=color, label=name)
ax2.set(xlabel='Round', ylabel='Test Accuracy', title='E1: FedAvg Homogeneous\n(same MediumCNN — WORKS)')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3); ax2.set_ylim(0, 1.05)

# Plot 3: E3 Distillation clients
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(rounds, distill_history['global_acc'], 'k-o', lw=2.5, ms=7, label='Global', zorder=5)
for i, (name, color) in enumerate(zip(cnames, colors_clients)):
    ax3.plot(rounds, [r[i] for r in distill_history['client_accs']], '--', lw=1.5, color=color, label=name)
ax3.set(xlabel='Round', ylabel='Test Accuracy', title='E3: Distillation Heterogeneous\n(different CNNs — WORKS)')
ax3.legend(fontsize=8); ax3.grid(alpha=0.3); ax3.set_ylim(0, 1.05)

# Plot 4: E2 FedAvg hetero crash visualization
ax4 = fig.add_subplot(gs[1, 0])
crash_rounds = [r for r,c in zip(rounds, fedavg_hetero_history['crashed']) if c]
ax4.bar(rounds, fedavg_hetero_history['global_acc'], color=[
    '#ff4444' if c else '#4444ff'
    for c in fedavg_hetero_history['crashed']], alpha=0.8)
ax4.set(xlabel='Round', ylabel='Global Accuracy', title=f'E2: FedAvg Heterogeneous\n💥 Crashed in {len(crash_rounds)}/{CONFIG["num_rounds"]} rounds')
ax4.set_ylim(0, 1.05); ax4.grid(alpha=0.3)
ax4.axhline(0.1, color='orange', linestyle='--', lw=2, label='Random baseline (10%)')
ax4.legend(fontsize=9)

# Plot 5: Per-client comparison E1 vs E3
ax5 = fig.add_subplot(gs[1, 1:])
x = np.arange(3); width = 0.35
fa_final = [fedavg_history['client_accs'][-1][i] for i in range(3)]
di_final = [distill_history['client_accs'][-1][i] for i in range(3)]
bars1 = ax5.bar(x - width/2, fa_final, width, label='E1: FedAvg (Homo)', color='#5b8db8', alpha=0.85)
bars2 = ax5.bar(x + width/2, di_final, width, label='E3: Distillation (Hetero)', color='#6bbf6a', alpha=0.85)
ax5.set(xlabel='Client', ylabel='Final Test Accuracy',
        title='Final Accuracy per Client\nE1 FedAvg (same model) vs E3 Distillation (different models)')
ax5.set_xticks(x); ax5.set_xticklabels(['Client 1\n(Small CNN)', 'Client 2\n(Medium CNN)', 'Client 3\n(Large CNN)'])
ax5.set_ylim(0, 1.15); ax5.legend(fontsize=10); ax5.grid(alpha=0.3, axis='y')
for bar in bars1: ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2: ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.savefig('hfl_cifar10_all_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Results saved as hfl_cifar10_all_experiments.png')

---
## Section 10: Final Comparison Table

In [ ]:
fa  = fedavg_history
fah = fedavg_hetero_history
di  = distill_history
crashes = sum(fah['crashed'])

print('='*72)
print('  FINAL RESULTS — CIFAR-10 Object Image Classification')
print('='*72)
print(f'  Dataset   : CIFAR-10 (60,000 real object images, 10 classes)')
print(f'  Framework : Flower v{fl.__version__} | Rounds: {CONFIG["num_rounds"]} | Non-IID α={CONFIG["non_iid_alpha"]}')
print()
print(f'  {"Metric":<35} {"E1 FedAvg":>12} {"E2 FedAvg":>12} {"E3 Distill":>12}')
print(f'  {"":<35} {"(Homo)":>12} {"(Hetero)"+" 💥":>12} {"(Hetero)"+" ✅":>12}')
print(f'  {"-"*72}')
print(f'  {"Global Model Accuracy":<35} {fa["global_acc"][-1]:>12.4f} {fah["global_acc"][-1]:>12.4f} {di["global_acc"][-1]:>12.4f}')
print(f'  {"Client 1 (Small CNN)":<35} {fa["client_accs"][-1][0]:>12.4f} {fah["client_accs"][-1][0]:>12.4f} {di["client_accs"][-1][0]:>12.4f}')
print(f'  {"Client 2 (Medium CNN)":<35} {fa["client_accs"][-1][1]:>12.4f} {fah["client_accs"][-1][1]:>12.4f} {di["client_accs"][-1][1]:>12.4f}')
print(f'  {"Client 3 (Large CNN)":<35} {fa["client_accs"][-1][2]:>12.4f} {fah["client_accs"][-1][2]:>12.4f} {di["client_accs"][-1][2]:>12.4f}')
print(f'  {"Avg Client Accuracy":<35} {np.mean(fa["client_accs"][-1]):>12.4f} {np.mean(fah["client_accs"][-1]):>12.4f} {np.mean(di["client_accs"][-1]):>12.4f}')
print(f'  {"-"*72}')
print(f'  {"FedAvg crashed rounds":<35} {"0/10":>12} {str(crashes)+"/10":>12} {"N/A":>12}')
print(f'  {"Supports asymmetric models":<35} {"❌ No":>12} {"❌ No":>12} {"✅ Yes":>12}')
print(f'  {"Shares raw data":<35} {"No":>12} {"No":>12} {"No":>12}')
print(f'  {"Aggregation":<35} {"Weight avg":>12} {"Weight avg":>12} {"Distillation":>12}')
print('='*72)
print()
print('KEY FINDING:')
print(f'  E1 FedAvg (homogeneous)       → {fa["global_acc"][-1]:.4f} ✅ works but unrealistic')
print(f'  E2 FedAvg (heterogeneous)     → {fah["global_acc"][-1]:.4f} 💥 crashes, cannot aggregate')
print(f'  E3 Distillation (heterogeneous) → {di["global_acc"][-1]:.4f} ✅ works with asymmetric models')
print()
gap = fa['global_acc'][-1] - di['global_acc'][-1]
print(f'  Accuracy cost of supporting heterogeneity: {gap:.4f} ({gap*100:.1f}%)')
print(f'  → This is the price of real-world deployability.')

---
## Section 11: Thesis Discussion

### Dataset Choice
**CIFAR-10** was chosen because:
- Real colour photographs (not synthetic) — airplanes, cars, animals, ships, trucks
- Standard FL benchmark — used in McMahan (2017), Lin (2020), and cited in arXiv:2307.10616
- More challenging than MNIST — colour, texture, background variation
- Same 10-class labeling across all clients — exactly as your professor specified

### What the 3 experiments prove
| Experiment | Finding |
|---|---|
| E1: FedAvg Homogeneous | High accuracy but requires identical models — unrealistic assumption |
| E2: FedAvg Heterogeneous | **Mathematically impossible** — crashes on every round |
| E3: Distillation Heterogeneous | Strong accuracy with fully asymmetric models — real-world solution |

### Mapped to survey arXiv:2307.10616
| Technique | Survey Section | Implementation |
|---|---|---|
| Statistical Heterogeneity | Sec 2.1 | Dirichlet Non-IID (α=0.5) |
| Model Heterogeneity | Sec 2.2 | Small/Medium/Large CNN |
| FedAvg Baseline | Sec 3.1 | E1 homogeneous experiment |
| Ensemble Distillation | Sec 3.2.2 | E3 FedDF-style aggregation |
| Federated Transfer Learning | Sec 3.3 | Soft label transfer server→clients |

### Say this to your professor
> *'I used CIFAR-10 — a real dataset of 60,000 colour object images across 10 classes including airplanes, vehicles, animals and trucks. I ran three controlled experiments: E1 shows FedAvg working under the unrealistic assumption that all clients share the same architecture. E2 mathematically proves FedAvg cannot handle heterogeneous models — it crashes every round due to weight shape mismatches. E3 demonstrates that ensemble knowledge distillation solves this — the same asymmetric clients collaborate successfully by sharing only soft predictions on a public proxy dataset, achieving strong accuracy without sharing raw data or requiring identical model architectures.'*

---
# PART 2 — Professor's Extra Task

Two additional heterogeneity dimensions, on top of the model-heterogeneity study above:

| Task | What changes across clients | What stays fixed |
|---|---|---|
| **Task 1 (Sec 12)** | Each client randomly loses a *different number* of input features (pixels) — client *i* removes `m_i` features | Same dataset (CIFAR-10), same model architecture |
| **Task 2 (Sec 13)** | Each client draws from a **totally different dataset** (CIFAR-10 vs STL-10) | Same label space (both remapped to a shared 9-class vocabulary), same model architecture |

Both experiments still use **FedAvg**, because in both cases the *model architecture* is identical across
clients (only the *data* differs) — so weight averaging is mathematically valid. This isolates the effect
of **statistical/feature heterogeneity** from the **model heterogeneity** studied in Part 1.

---
## Section 12: Task 1 — Same Dataset, Random Feature Removal per Client

**Setup.** All `N` clients still train on CIFAR-10 with the *same* CNN architecture. But each client `i`
has a sensor/device that only captures part of the image: we randomly zero out `m_i` of the 3,072 pixel
features (32×32×3) for that client, where `m_1, m_2, ..., m_N` are different amounts (e.g. 0%, 15%, 30%, 50%
of pixels missing). This simulates faulty sensors, occlusion, or missing channels — a very common
real-world feature-heterogeneity scenario (distinct from label heterogeneity / Non-IID class skew).

Because the *architecture* is unchanged, FedAvg should still run without crashing — but we expect accuracy
to degrade as `m_i` grows, and we can measure how much each client's missing-feature rate drags down both
its own performance and the shared global model.

In [ ]:
CONFIG_FEAT = {
    'num_clients_feat'   : 4,
    'num_rounds_feat'    : 8,
    'local_epochs_feat'  : 4,
    'batch_size_feat'    : 64,
    'learning_rate_feat' : 0.001,
    # m_i as a FRACTION of the 3,072 pixel-features (32*32*3) removed for client i
    'feature_removal_fracs': [0.00, 0.15, 0.30, 0.50],
}
TOTAL_FEATURES = 32 * 32 * 3
assert len(CONFIG_FEAT['feature_removal_fracs']) == CONFIG_FEAT['num_clients_feat']

print('Configuration loaded')
for k, v in CONFIG_FEAT.items():
    print(f'   {k}: {v}')
print(f'   total pixel-features per image: {TOTAL_FEATURES}')

In [ ]:
np.random.seed(123)

# Raw (untransformed) CIFAR-10 so we can attach a DIFFERENT transform+mask per client
train_data_raw_feat = torchvision.datasets.CIFAR10('.', train=True, download=True)

# Non-IID class split across clients (reuse the Dirichlet splitter from Section 3)
client_indices_feat = dirichlet_split(train_data_raw_feat, CONFIG_FEAT['num_clients_feat'],
                                       CONFIG['non_iid_alpha'])

# For every client, sample which m_i pixel-features get zeroed out (fixed per client for all its images)
feature_mask_idx = []
for i, frac in enumerate(CONFIG_FEAT['feature_removal_fracs']):
    m_i = int(round(frac * TOTAL_FEATURES))
    rng = np.random.RandomState(1000 + i)
    idx = rng.choice(TOTAL_FEATURES, size=m_i, replace=False) if m_i > 0 else np.array([], dtype=int)
    feature_mask_idx.append(idx)
    print(f'   Client {i+1}: removing m_{i+1} = {m_i} / {TOTAL_FEATURES} features '
          f'({frac*100:.0f}%), {len(client_indices_feat[i])} samples')

class FeatureMaskedCIFAR(torch.utils.data.Dataset):
    """Wraps CIFAR-10 and zeroes out a fixed random set of pixel-features per client."""
    def __init__(self, base_dataset, indices, transform, mask_idx):
        self.base, self.indices, self.transform, self.mask_idx = base_dataset, indices, transform, mask_idx
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        img, label = self.base[self.indices[i]]
        x = self.transform(img)
        if len(self.mask_idx):
            flat = x.reshape(-1)
            flat[self.mask_idx] = 0.0
            x = flat.reshape(x.shape)
        return x, label

client_loaders_feat = [
    DataLoader(
        FeatureMaskedCIFAR(train_data_raw_feat, client_indices_feat[i], transform_train, feature_mask_idx[i]),
        batch_size=CONFIG_FEAT['batch_size_feat'], shuffle=True
    )
    for i in range(CONFIG_FEAT['num_clients_feat'])
]
print('\nFeature-heterogeneous client loaders ready')

In [ ]:
print('='*60)
print('  E4: FedAvg — Feature Heterogeneity (random features removed per client)')
print('='*60)

# Same architecture for every client -> FedAvg weight averaging is valid
feat_clients = [MediumCNN().to(DEVICE) for _ in range(CONFIG_FEAT['num_clients_feat'])]
feat_opts    = [optim.Adam(m.parameters(), lr=CONFIG_FEAT['learning_rate_feat']) for m in feat_clients]
feat_global  = MediumCNN().to(DEVICE)
feat_history = {'round': [], 'global_acc': [], 'client_accs': []}

for rnd in range(1, CONFIG_FEAT['num_rounds_feat'] + 1):
    g_state = feat_global.state_dict()
    for c in feat_clients:
        c.load_state_dict(g_state)

    for c, opt, loader in zip(feat_clients, feat_opts, client_loaders_feat):
        train_local(c, loader, opt, CONFIG_FEAT['local_epochs_feat'], DEVICE)

    total = sum(len(client_indices_feat[i]) for i in range(CONFIG_FEAT['num_clients_feat']))
    avg_state = {}
    for key in feat_clients[0].state_dict():
        avg_state[key] = sum(
            feat_clients[i].state_dict()[key].float() * (len(client_indices_feat[i]) / total)
            for i in range(CONFIG_FEAT['num_clients_feat'])
        )
    feat_global.load_state_dict(avg_state)

    g_acc  = evaluate_model(feat_global, test_loader, DEVICE)
    c_accs = [evaluate_model(c, test_loader, DEVICE) for c in feat_clients]
    feat_history['round'].append(rnd)
    feat_history['global_acc'].append(g_acc)
    feat_history['client_accs'].append(c_accs)
    print(f'  Round {rnd:02d} | Global: {g_acc:.4f} | Clients: {[f"{a:.4f}" for a in c_accs]}')

print(f'\nE4 Final Global Accuracy: {feat_history["global_acc"][-1]:.4f}')
print('   (FedAvg does NOT crash here, unlike E2 — architecture is identical.')
print('    Accuracy loss instead comes from missing features per client.)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('E4: FedAvg under Feature Heterogeneity (Task 1)', fontsize=13, fontweight='bold')

rounds_feat = list(range(1, CONFIG_FEAT['num_rounds_feat'] + 1))
axes[0].plot(rounds_feat, feat_history['global_acc'], 'k-o', lw=2.5, ms=7, label='Global', zorder=5)
colors_feat = plt.cm.viridis(np.linspace(0, 0.85, CONFIG_FEAT['num_clients_feat']))
for i, frac in enumerate(CONFIG_FEAT['feature_removal_fracs']):
    axes[0].plot(rounds_feat, [r[i] for r in feat_history['client_accs']], '--', lw=1.5,
                 color=colors_feat[i], label=f'Client {i+1} (m={frac*100:.0f}% removed)')
axes[0].set(xlabel='Round', ylabel='Test Accuracy', title='Accuracy over Rounds')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

final_accs = feat_history['client_accs'][-1]
axes[1].bar([f'{f*100:.0f}%' for f in CONFIG_FEAT['feature_removal_fracs']], final_accs, color=colors_feat)
axes[1].axhline(feat_history['global_acc'][-1], color='red', ls='--', lw=2, label='Global model')
axes[1].set(xlabel='Fraction of features removed (m_i)', ylabel='Final Test Accuracy',
            title='Accuracy vs. Missing-Feature Rate')
axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('e4_feature_heterogeneity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved as e4_feature_heterogeneity.png')

---
# PART 3 — Comparing 6 Federated Learning Algorithms (One Dataset)

Per your professor's latest instruction: **one dataset only** (CIFAR-10, the same Non-IID 3-client split
from Section 3), **same model architecture for every client** (`MediumCNN`, so every algorithm is being
compared fairly on identical ground), and **6 different federated algorithms**:

| # | Algorithm | Idea |
|---|---|---|
| 1 | **FedAvg** | Plain weighted average of client weights (baseline) |
| 2 | **FedProx** | FedAvg + a penalty term that stops local models drifting too far from the global model |
| 3 | **FedAdagrad** | Server treats the aggregated update as a gradient and applies Adagrad (adaptive, accumulating) |
| 4 | **FedYogi** | Server-side adaptive optimizer, more stable than Adagrad on Non-IID data |
| 5 | **FedAdam** | Server-side Adam optimizer applied to the aggregated update |
| 6 | **FedDistillation (FedDF)** | Clients don't share weights at all — they share *predictions* on a small public proxy set, and the server distills those into the global model |

FedAvg, FedProx, FedAdagrad, FedYogi and FedAdam all still average/combine **model weights**, so they need
identical architectures — satisfied here. FedDistillation doesn't need that at all (it would even work with
different architectures), but we keep it on the same architecture too so the comparison is fair.

In [ ]:
CONFIG_CMP = {
    'num_rounds'    : CONFIG['num_rounds'],      # reuse Section 2 settings -> same client split, same rounds
    'local_epochs'  : CONFIG['local_epochs'],
    'learning_rate' : CONFIG['learning_rate'],
    'mu_prox'       : 0.01,     # FedProx proximal strength
    'server_lr'     : 0.1,      # server-side learning rate for FedAdagrad / FedYogi / FedAdam
    'beta1'         : 0.9,      # momentum term (FedYogi / FedAdam)
    'beta2'         : 0.99,     # 2nd-moment term (FedAdagrad / FedYogi / FedAdam)
    'tau'           : 1e-3,     # numerical stability term
}
print('Using the SAME CIFAR-10 Non-IID client split as Section 3 (client_loaders, 3 clients).')
print('Using the SAME MediumCNN architecture for every client, in every algorithm.')
for k, v in CONFIG_CMP.items():
    print(f'   {k}: {v}')

In [ ]:
def train_local_fedprox(model, loader, optimizer, epochs, device, global_state, mu):
    """Same as train_local, but adds the FedProx proximal term (mu/2)*||w_local - w_global||^2."""
    model.train()
    ce = nn.CrossEntropyLoss()
    global_params = {k: v.clone().detach().to(device) for k, v in global_state.items()}
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            prox = sum(((p - global_params[n]) ** 2).sum()
                       for n, p in model.named_parameters() if n in global_params)
            (loss + (mu / 2) * prox).backward()
            optimizer.step()

def weighted_avg_state(models, sizes):
    """Plain FedAvg-style weighted average of state_dicts."""
    total = sum(sizes)
    keys = models[0].state_dict().keys()
    return {k: sum(models[i].state_dict()[k].float() * (sizes[i] / total) for i in range(len(models)))
            for k in keys}

def fedopt_server_update(global_model, avg_state, server_state, algo, cfg):
    """
    FedOpt-style server update (Reddi et al., 2021).
    delta = avg_client_weights - global_weights  (the 'pseudo-gradient', with a sign flip)
    algo in {'fedavg', 'fedadagrad', 'fedyogi', 'fedadam'}
    server_state carries the running m (momentum) and v (2nd moment) per parameter tensor.
    """
    g_state = global_model.state_dict()
    new_state = {}
    for key in g_state:
        delta = avg_state[key].float() - g_state[key].float()
        if algo == 'fedavg':
            new_state[key] = g_state[key].float() + delta          # = avg_state, plain FedAvg
            continue
        if key not in server_state:
            server_state[key] = {'m': torch.zeros_like(delta), 'v': torch.full_like(delta, cfg['tau'] ** 2)}
        m, v = server_state[key]['m'], server_state[key]['v']
        if algo == 'fedadagrad':
            v = v + delta ** 2
        elif algo == 'fedyogi':
            m = cfg['beta1'] * m + (1 - cfg['beta1']) * delta
            v = v - (1 - cfg['beta2']) * torch.sign(v - delta ** 2) * (delta ** 2)
        elif algo == 'fedadam':
            m = cfg['beta1'] * m + (1 - cfg['beta1']) * delta
            v = cfg['beta2'] * v + (1 - cfg['beta2']) * (delta ** 2)
        else:
            raise ValueError(algo)
        update = m if algo != 'fedadagrad' else delta
        server_state[key] = {'m': m, 'v': v}
        new_state[key] = g_state[key].float() + cfg['server_lr'] * update / (torch.sqrt(v) + cfg['tau'])
    return new_state

print('Local-training and server-aggregation helper functions ready.')

In [ ]:
def run_weight_based_algo(algo_name, use_prox=False):
    """Runs FedAvg / FedProx / FedAdagrad / FedYogi / FedAdam — all weight-averaging algorithms,
    on the identical CIFAR-10 Non-IID client split and MediumCNN architecture."""
    print('='*60); print(f'  {algo_name.upper()}'); print('='*60)

    clients = [MediumCNN().to(DEVICE) for _ in range(CONFIG['num_clients'])]
    opts    = [optim.Adam(m.parameters(), lr=CONFIG_CMP['learning_rate']) for m in clients]
    global_model = MediumCNN().to(DEVICE)
    server_state = {}
    sizes = [len(idx) for idx in client_indices]
    history = {'round': [], 'global_acc': []}

    for rnd in range(1, CONFIG_CMP['num_rounds'] + 1):
        g_state = global_model.state_dict()
        for c in clients:
            c.load_state_dict(g_state)

        for c, opt, loader in zip(clients, opts, client_loaders):
            if use_prox:
                train_local_fedprox(c, loader, opt, CONFIG_CMP['local_epochs'], DEVICE, g_state, CONFIG_CMP['mu_prox'])
            else:
                train_local(c, loader, opt, CONFIG_CMP['local_epochs'], DEVICE)

        avg_state = weighted_avg_state(clients, sizes)
        new_state = fedopt_server_update(global_model, avg_state, server_state, algo_name, CONFIG_CMP)
        global_model.load_state_dict(new_state)

        acc = evaluate_model(global_model, test_loader, DEVICE)
        history['round'].append(rnd); history['global_acc'].append(acc)
        print(f'  Round {rnd:02d} | Global Accuracy: {acc:.4f}')
    print(f'Final: {history["global_acc"][-1]:.4f}\n')
    return history

# 1. FedAvg
history_fedavg_cmp     = run_weight_based_algo('fedavg')
# 2. FedProx  (aggregation = plain FedAvg update, local objective has the proximal penalty)
history_fedprox_cmp    = run_weight_based_algo('fedavg', use_prox=True)
# 3. FedAdagrad
history_fedadagrad_cmp = run_weight_based_algo('fedadagrad')
# 4. FedYogi
history_fedyogi_cmp    = run_weight_based_algo('fedyogi')
# 5. FedAdam
history_fedadam_cmp    = run_weight_based_algo('fedadam')

In [ ]:
# 6. FedDistillation (FedDF) — clients share PREDICTIONS on a small public proxy set, not weights.
print('='*60); print('  FEDDISTILLATION (FedDF)'); print('='*60)

clients_fd = [MediumCNN().to(DEVICE) for _ in range(CONFIG['num_clients'])]
opts_fd    = [optim.Adam(m.parameters(), lr=CONFIG_CMP['learning_rate']) for m in clients_fd]
global_fd  = MediumCNN().to(DEVICE)
sizes      = [len(idx) for idx in client_indices]
public_X_fd, _ = next(iter(public_loader))
history_feddistill_cmp = {'round': [], 'global_acc': []}

for rnd in range(1, CONFIG_CMP['num_rounds'] + 1):
    # 1) each client trains locally on its own private data only
    for c, opt, loader in zip(clients_fd, opts_fd, client_loaders):
        train_local(c, loader, opt, CONFIG_CMP['local_epochs'], DEVICE)

    # 2) server collects each client's SOFT PREDICTIONS on the shared public proxy set
    client_soft_preds = [get_soft_preds(c, public_X_fd, DEVICE) for c in clients_fd]
    weights = np.array(sizes) / sum(sizes)
    ensemble_soft = sum(w * p for w, p in zip(weights, client_soft_preds))   # weighted ensemble = teacher

    # 3) global model distills the ensemble's knowledge (no weight averaging at all)
    global_opt = optim.Adam(global_fd.parameters(), lr=CONFIG_CMP['learning_rate'])
    for _ in range(CONFIG['distill_epochs']):
        global_opt.zero_grad()
        logits = global_fd(public_X_fd.to(DEVICE))
        T = CONFIG['temperature']
        s_log  = F.log_softmax(logits / T, dim=1)
        t_soft = F.softmax(ensemble_soft.to(DEVICE) / T, dim=1)
        (nn.KLDivLoss(reduction='batchmean')(s_log, t_soft) * (T ** 2)).backward()
        global_opt.step()

    # 4) broadcast the newly-distilled global model back down so next round's local training starts from it
    for c in clients_fd:
        c.load_state_dict(global_fd.state_dict())

    acc = evaluate_model(global_fd, test_loader, DEVICE)
    history_feddistill_cmp['round'].append(rnd)
    history_feddistill_cmp['global_acc'].append(acc)
    print(f'  Round {rnd:02d} | Global Accuracy: {acc:.4f}')

print(f'Final: {history_feddistill_cmp["global_acc"][-1]:.4f}')

In [ ]:
results_cmp = {
    'FedAvg'          : history_fedavg_cmp,
    'FedProx'         : history_fedprox_cmp,
    'FedAdagrad'      : history_fedadagrad_cmp,
    'FedYogi'         : history_fedyogi_cmp,
    'FedAdam'         : history_fedadam_cmp,
    'FedDistillation' : history_feddistill_cmp,
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('6 Federated Algorithms on ONE Dataset (CIFAR-10, same clients, same architecture)',
             fontsize=13, fontweight='bold')

palette = plt.cm.tab10(np.linspace(0, 1, 6))
for (name, hist), color in zip(results_cmp.items(), palette):
    axes[0].plot(hist['round'], hist['global_acc'], '-o', lw=2, ms=5, color=color, label=name)
axes[0].set(xlabel='Round', ylabel='Global Test Accuracy', title='Accuracy over Rounds')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.05)

final_accs = [hist['global_acc'][-1] for hist in results_cmp.values()]
bars = axes[1].bar(list(results_cmp.keys()), final_accs, color=palette)
axes[1].set(ylabel='Final Global Test Accuracy', title='Final Accuracy Comparison')
axes[1].tick_params(axis='x', rotation=25); axes[1].grid(alpha=0.3, axis='y')
for b, a in zip(bars, final_accs):
    axes[1].text(b.get_x() + b.get_width()/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('e6_six_algo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved as e6_six_algo_comparison.png')

print('\nFinal ranking:')
for name, acc in sorted(zip(results_cmp.keys(), final_accs), key=lambda x: -x[1]):
    print(f'   {name:18s}: {acc:.4f}')

---
## Section 17: Summary — 6 Algorithms, 1 Dataset

**What's identical across all 6 runs:** dataset (CIFAR-10), client data split (same Non-IID Dirichlet split
from Section 3), model architecture (`MediumCNN` for every client), number of rounds, and local epochs. The
**only** thing that changes between runs is the federated algorithm itself — so any accuracy difference you
see is caused by the algorithm, not by a different setup.

**How to describe each one to your professor, in plain words:**
- **FedAvg** — every client sends its trained model back, the server just averages them by how much data each client has.
- **FedProx** — same as FedAvg, but while training locally each client is penalized for straying too far from the global model — this helps a lot when clients have very different (Non-IID) data.
- **FedAdagrad** — the server treats the "distance moved" as a gradient and applies Adagrad, so parameters that keep changing a lot get smaller updates over time.
- **FedYogi** — similar server-side idea but a more controlled/stable version of Adagrad, usually better for Non-IID data.
- **FedAdam** — the server applies the well-known Adam optimizer to the aggregated update instead of just adding it directly.
- **FedDistillation (FedDF)** — the most different one: clients never send their weights at all. They only send their *predictions* on a small shared public dataset, and the server trains the global model to imitate the combined ("ensembled") predictions. This is the only method here that would still work even if clients had different model architectures.

The bar chart above shows which algorithm reached the highest final accuracy on this dataset/split — that
ranking is what you report to your professor, along with the plain-word description of why each algorithm
behaves the way it does.